# Week 2 Presentation Brief — Variation A
## 🦠 Bacterial Growth: The Hospital Infection Challenge
**SCIE1500 — Semester 2, 2026 | Group 1 — presenting in Week 3**

> Work through all parts during the Week 2 lab. Your **10-minute Week 3 presentation** should cover: the problem, your model, key results, and policy implications.
> Do not show raw code in your presentation — show graphs and equations.

---
## 📋 Scenario

You are a consultant for **Royal Perth Hospital's** infection control team. A strain of antibiotic-resistant *Staphylococcus aureus* (MRSA) has been detected on a surgical instrument.

- Doubling time at room temperature: **25 minutes**
- Initial contamination: **800 cells**
- **Hospital policy:** If count exceeds **1,000,000 cells**, the surgical suite shuts down for sterilization — cost: $50,000 + 48-hour delay.

---
## 🎯 Your Task

| Part | Topic | Time |
|------|-------|------|
| A | Build growth models in two forms; find shutdown time | ~20 min |
| B | Model the effect of refrigeration on bacterial growth | ~15 min |
| C | Make a sterilisation decision and write a policy recommendation | ~10 min |
| D | Verify the Rule of 70 for this scenario | ~10 min |

### Getting started — what does the model look like?

We're modelling MRSA growing exponentially from an initial count of 800 cells. Because we know the **doubling time** (25 minutes at room temperature), we can write:

$$N(t) = 800 \times 2^{t/25}$$

or equivalently, converting to the continuous-rate form with $k = \ln(2)/25$:

$$N(t) = 800 \, e^{kt}$$

Run the setup cell to define these parameters and check what $k$ looks like numerically.

Setup — run first.

In [ ]:
# Setup — run first
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

N0        = 800
T_room    = 25           # doubling time at room temperature (minutes)
threshold = 1_000_000    # shutdown threshold (cells)

k_room = np.log(2) / T_room
print("k_room =", k_room)           # raw value — continuous growth rate per minute
print("N0 =", N0, "cells")
print("Shutdown threshold:", threshold, "cells")

---
## Part A: Modeling Bacterial Growth (~20 min)

We'll write the growth model in both forms and verify they give identical results, then find when the population hits the shutdown threshold.

### A.1 — Two equivalent forms of the model

Define both forms as Python functions, then evaluate them at 1, 2, and 3 hours to confirm they agree. Note: at room temperature, each 25-minute interval is one doubling — so after 1 hour (60 min) there are $60/25 = 2.4$ doublings.

Both forms of the growth model.

In [ ]:
# A.1 — Both forms of the growth model
def N_v1(t): return N0 * 2**(t / T_room)         # doubling form
def N_v2(t): return N0 * np.exp(k_room * t)      # continuous-rate form

print("Verifying both forms agree:")
for t_h in [1, 2, 3]:
    t = t_h * 60
    print(f"  t = {t_h}h ({t} min):  2^(t/T) = {N_v1(t):.0f}   e^(kt) = {N_v2(t):.0f}")

### A.2 — Finding the shutdown time

We need to find when $N(t) = 1{,}000{,}000$. Starting from the doubling form:
$$800 \times 2^{t/25} = 1{,}000{,}000$$

Taking $\log_2$ of both sides and solving for $t$ gives $t = 25 \times \log_2(1{,}000{,}000 / 800)$.

After computing the shutdown time, we'll plot the full growth curve on a **log scale** — this makes exponential growth appear as a straight line and makes the threshold easy to read off.

Time to reach shutdown threshold.

In [ ]:
# A.2 — Time to reach shutdown threshold
t_shutdown = T_room * np.log2(threshold / N0)
print(f"Time to shutdown threshold: {t_shutdown:.1f} minutes = {t_shutdown / 60:.2f} hours")

t_vals = np.linspace(0, 350, 600)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_vals, N_v1(t_vals), "steelblue", lw=2, label="MRSA count (room temp)")
ax.axhline(threshold, color="red",    ls="--", lw=1.5, label=f"Shutdown threshold (1M cells)")
ax.axvline(t_shutdown, color="orange", ls=":",  lw=1.5, label=f"Threshold at {t_shutdown:.0f} min")
ax.set_yscale("log")
ax.set_xlabel("Time (minutes)")
ax.set_ylabel("Bacterial count (log scale)")
ax.set_title("MRSA Growth — Royal Perth Hospital")
ax.legend()
plt.tight_layout()
plt.show()

✏️ **Group discussion A:**
1. Express the shutdown time as "X hours and Y minutes". If contamination is detected at 8:00 AM, what is the latest safe action time?
2. Why does the log scale make this plot more informative than a linear scale?

```python
# Answers:
# 1. Hours and minutes:
hrs = int(t_shutdown // 60)
mins = t_shutdown % 60
print(f"Shutdown at {hrs}h {mins:.0f}min")
# 2. Why log scale:
#    ...
```

---
## Part B: Effect of Refrigeration (~15 min)

At **4°C**, bacterial growth slows significantly — the doubling time increases from 25 minutes to **180 minutes**. We model this as a separate exponential function with a smaller $k$. The question for the hospital: does refrigeration buy enough time to make a considered decision?

### B.1 — Comparing room temperature vs refrigerated growth

We'll compute the new $k$ for 4°C, plot both growth curves on the same axes, and calculate the shutdown time for each condition. The ratio of shutdown times tells us directly how much time refrigeration buys.

Cold storage model (4°C, doubling time = 180 min).

In [ ]:
# B.1 — Cold storage model (4°C, doubling time = 180 min)
T_cold = 180
k_cold = np.log(2) / T_cold

def N_cold(t): return N0 * np.exp(k_cold * t)

t_shutdown_cold = T_cold * np.log2(threshold / N0)
print(f"Room temp: shutdown at {t_shutdown:.1f} min ({t_shutdown / 60:.2f} h)")
print(f"At 4°C:    shutdown at {t_shutdown_cold:.0f} min ({t_shutdown_cold / 60:.1f} h)")
print(f"Refrigeration buys {t_shutdown_cold / t_shutdown:.1f}x more time")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t_vals, N_v1(t_vals),   "steelblue", lw=2, label=f"Room temp (T = {T_room} min)")
ax.plot(t_vals, N_cold(t_vals), "seagreen",  lw=2, label=f"4°C refrigerated (T = {T_cold} min)")
ax.axhline(threshold, color="red", ls="--", label="Shutdown threshold")
ax.set_yscale("log")
ax.set_xlabel("Time (minutes)")
ax.set_ylabel("Bacterial count (log scale)")
ax.set_title("MRSA Growth: Room Temp vs Refrigeration")
ax.legend()
plt.tight_layout()
plt.show()

✏️ **Group discussion B:**
1. Can the hospital safely refrigerate overnight (8 hours = 480 min) before deciding?
2. What does this imply for hospital protocols about instrument handling?

```python
# Answers:
# 1. Count after 8h refrigeration:
print("Count after 8h cold:", N_cold(480))
print("Still below threshold?", N_cold(480) < threshold)
# 2. Protocol implication:
#    ...
```

---
## Part C: Sterilization Decision (~10 min)

The hospital administrator faces a choice:
- **Option 1:** Immediate sterilization — $50,000 + 48-hour delay
- **Option 2:** Refrigerate for 12 hours, re-test, then decide

Your job is to determine what the bacterial count will be after 12 hours of refrigeration, then make a data-backed recommendation.

In [ ]:
# C.1 — Bacterial count after 12 hours of refrigeration
t_12h = 12 * 60       # convert to minutes
N_after = N_cold(t_12h)

print("Count after 12h refrigeration:", round(N_after))
print("Still below threshold?", N_after < threshold)
print(f"Safety margin: {threshold / N_after:.1f}x below threshold")

✏️ **Policy recommendation (write as a group):**

Write 3–4 sentences recommending Option 1 or Option 2 to the hospital administrator. Include:
- Which option you recommend and why
- Specific numbers supporting the recommendation
- One key uncertainty or limitation in your model

```
RECOMMENDATION:
...
```

---
## Part D: Rule of 70 Verification (~10 min)

We claimed the doubling time is 25 minutes. But the Rule of 70 is an *approximation* based on the percentage growth rate. How well does it hold up for bacterial growth?

The key step: find the exact per-minute percentage growth rate from $k$, then apply $T \approx 70/r$.

In [ ]:
# D.1 — Exact % growth per minute vs Rule of 70 prediction
r_per_min_pct = (np.exp(k_room) - 1) * 100    # exact per-minute % rate

T_rule70 = 70 / r_per_min_pct

print(f"Exact % growth per minute:  {r_per_min_pct:.4f}%")
print(f"Rule of 70 doubling time:   {T_rule70:.2f} min")
print(f"Actual doubling time:       {T_room} min")
print(f"Error: {abs(T_rule70 - T_room) / T_room * 100:.1f}%")
print()
print("Note: the Rule of 70 works best for small percentage rates per period.")
print("Per-minute bacterial growth is large, so the approximation is less accurate.")

---
## ✅ Presentation Checklist (Week 3, 10 minutes)

1. **The Problem** (~2 min): What question were you answering?
2. **The Model** (~3 min): Why exponential? Show the equation and what $k$ means physically.
3. **Key Results** (~3 min): Shutdown times at room temp and refrigerated — show the comparison plot.
4. **Implications** (~2 min): What should hospital administrators take away?

> **Tips:** Speak to numbers — "the hospital has X hours" is more compelling than "t equals...". Practice timing.